<a href="https://colab.research.google.com/github/szymonbloch/tuberculosis_detection/blob/main/3_layer_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data preparing

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import joblib
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# ==========================================
# 1. WCZYTANIE ETYKIET Z PLIKÓW TXT
# ==========================================
POS_TXT_PATH = '/content/drive/MyDrive/NTWI_project/data/positive.txt'
NEG_TXT_PATH = '/content/drive/MyDrive/NTWI_project/data/negative.txt'

with open(POS_TXT_PATH, 'r') as f:
    pos_slides = [line.strip() for line in f if line.strip()]

with open(NEG_TXT_PATH, 'r') as f:
    neg_slides = [line.strip() for line in f if line.strip()]

# Budujemy słownik mapujący: {nazwa_slajdu: 0 lub 1}
labels_dict = {}
for slide in pos_slides:
    labels_dict[slide] = 1
for slide in neg_slides:
    labels_dict[slide] = 0

df_final_3l = pd.DataFrame(list(labels_dict.items()), columns=['image_name', 'image_output'])

In [ ]:
print(df_final_3l.head())

               image_name  image_output
0  DigitalSlide_B1M_10S_1             1
1  DigitalSlide_B1M_11S_1             1
2  DigitalSlide_B1M_13S_1             1
3  DigitalSlide_B1M_15S_1             1
4  DigitalSlide_B1M_16S_1             1


In [ ]:
# ==========================================
# 2. DETEKTOR ANOMALII (Połączenie 3 warstw za pomocą MAX) Wybór warstwy gdzie prawdopodobieństwo jest największe
# ==========================================
ANOMALY_3L_PATH = '/content/drive/MyDrive/NTWI_project/data/anomaly_detector/3-layer_ad_cnn_04_06_2025_00_31_21/*.csv'
anomaly_files = glob.glob(ANOMALY_3L_PATH)
table_list_anomaly = []

for file in anomaly_files:
    basename = os.path.basename(file)
    if '_Wholeslide_Default_' in basename:

      # Wyciągamy nazwę pacjenta oraz warstwę z nazwy pliku
      slide_name = basename.split('_Wholeslide_Default_')[0]
      layer = basename.split('_Wholeslide_Default_')[1].replace('.csv', '')

      # Ignorujemy warstwę Extended, bo chcemy sami zasymulować pooling 3D z warstw -0.8, 0, 0.8
      if layer != 'Extended':

        df_temp = pd.read_csv(file)
        df_temp['probability'] = df_temp['probability'].astype(str).str.extract(r'tensor\(([0-9.]+)').astype(float)

        # Wyciągamy współrzędne kafelka (np. 147_208)
        df_temp['tile_coord'] = df_temp['image_name'].astype(str).str.split('.').str[0].str.strip()

        # Tworzymy unikalny, globalny klucz kafelka
        df_temp['tile_name'] = slide_name + '_' + df_temp['tile_coord']
        df_temp['image_name'] = slide_name

        df_temp = df_temp[['tile_name', 'image_name', 'probability', 'output']]
        table_list_anomaly.append(df_temp)

df_anomaly_3l_raw = pd.concat(table_list_anomaly, ignore_index=True)

# Spłaszczanie w pionie: bierzemy maksimum z 3 warstw ostrości dla danego punktu XY pacjenta
df_anomaly_3l = df_anomaly_3l_raw.groupby(['tile_name', 'image_name']).agg({
    'probability': 'max',
    'output': 'max'
}).reset_index()
df_anomaly_3l.rename(columns={'output': 'tile_output'}, inplace=True)

In [ ]:
print(df_anomaly_3l)

                             tile_name              image_name  probability  \
0       DigitalSlide_B1M_10S_1_100_122  DigitalSlide_B1M_10S_1       0.0016   
1       DigitalSlide_B1M_10S_1_100_123  DigitalSlide_B1M_10S_1       0.0033   
2       DigitalSlide_B1M_10S_1_100_124  DigitalSlide_B1M_10S_1       0.0024   
3       DigitalSlide_B1M_10S_1_100_125  DigitalSlide_B1M_10S_1       0.0027   
4       DigitalSlide_B1M_10S_1_100_126  DigitalSlide_B1M_10S_1       0.0029   
...                                ...                     ...          ...   
806739     DigitalSlide_B1M_9S_1_9_133   DigitalSlide_B1M_9S_1       0.0035   
806740     DigitalSlide_B1M_9S_1_9_134   DigitalSlide_B1M_9S_1       0.0041   
806741     DigitalSlide_B1M_9S_1_9_135   DigitalSlide_B1M_9S_1       0.0040   
806742     DigitalSlide_B1M_9S_1_9_136   DigitalSlide_B1M_9S_1       0.0048   
806743     DigitalSlide_B1M_9S_1_9_137   DigitalSlide_B1M_9S_1       0.0013   

        tile_output  
0                 0  
1      

In [ ]:
COLORS_3L_PATH = '/content/drive/MyDrive/NTWI_project/data/color_features/3_layers/*.csv'
color_files = glob.glob(COLORS_3L_PATH)
table_list_colors = []

for file in color_files:
    basename = os.path.basename(file)

    if '_Wholeslide_Default_' in basename:
        slide_name = basename.split('_Wholeslide_Default_')[0]

        df_temp = pd.read_csv(file, header=None, sep=None, engine='python')
        df_temp.rename(columns={0: 'tile_coord'}, inplace=True)
        df_temp['tile_coord'] = df_temp['tile_coord'].astype(str).str.split('.').str[0].str.strip()

        # Tworzymy taki sam unikalny klucz jak w detektorze anomalii
        df_temp['tile_name'] = slide_name + '_' + df_temp['tile_coord']
        df_temp.drop(columns=['tile_coord'], inplace=True)

        table_list_colors.append(df_temp)

df_colors_3l_raw = pd.concat(table_list_colors, ignore_index=True)

# USUWANIE ZŁOŚLIWEJ KOLUMNY 59
df_colors_3l_raw = df_colors_3l_raw.dropna(axis=1, how='all')
if 59 in df_colors_3l_raw.columns:
    df_colors_3l_raw = df_colors_3l_raw.drop(columns=[59])
if '59' in df_colors_3l_raw.columns:
    df_colors_3l_raw = df_colors_3l_raw.drop(columns=['59'])

# Uśredniamy kolory
df_colors_3l = df_colors_3l_raw.groupby('tile_name').mean().reset_index()

In [ ]:
print(df_colors_3l)

                             tile_name          1      2            3  \
0       DigitalSlide_B1M_10S_1_100_122  21.750000  255.0  12263752.25   
1       DigitalSlide_B1M_10S_1_100_123  14.500000  255.0  11240716.25   
2       DigitalSlide_B1M_10S_1_100_124   0.000000  255.0  11548226.75   
3       DigitalSlide_B1M_10S_1_100_125  37.750000  255.0  11223052.50   
4       DigitalSlide_B1M_10S_1_100_126   0.000000  255.0  10989492.50   
...                                ...        ...    ...          ...   
822422     DigitalSlide_B1M_9S_1_9_133   0.000000  255.0  10938107.25   
822423     DigitalSlide_B1M_9S_1_9_134   0.000000  255.0  10539075.00   
822424     DigitalSlide_B1M_9S_1_9_135   0.000000  255.0  10832491.50   
822425     DigitalSlide_B1M_9S_1_9_136   0.000000  255.0  10550785.25   
822426     DigitalSlide_B1M_9S_1_9_137   3.666667  255.0  12355993.00   

               4             5        6            7        8         9  ...  \
0       50176.00  1.233243e+07  50176.0  12

# Data merging

In [ ]:
df_step1_3l = df_anomaly_3l.merge(df_final_3l, on='image_name', how='inner')
df_merged_3l = df_step1_3l.merge(df_colors_3l, on='tile_name', how='inner')
df_merged_3l.to_csv('/content/drive/MyDrive/NTWI_project/results/df_merged_3l.csv', index=False)

# Statistics from merged data

In [ ]:
import pandas as pd
import joblib
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

MERGED_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged_3l.csv'

df_merged_3l = pd.read_csv(MERGED_DATAFRAME_PATH)

# ==========================================
# 1. PRZYGOTOWANIE SŁOWNIKA AGREGACJI
# ==========================================
# Sprawdzamy nagłówek etykiety (obsługa potencjalnych różnic w nazewnictwie)
if 'image_output' in df_merged_3l.columns and 'output_image' not in df_merged_3l.columns:
    df_merged_3l = df_merged_3l.rename(columns={'image_output': 'output_image'})

statistic_dict = {
    'probability': ['mean', 'max', 'std'],
    'output_image': ['first']
}

# Automatyczne dodanie wszystkich kolumn z kolorami (0, 1, 2... 58) do słownika
for col in df_merged_3l.columns:
    if str(col).isdigit():
        statistic_dict[col] = ['mean']

In [ ]:
# ==========================================
# 2. GENEROWANIE STATYSTYK PACJENTA
# ==========================================
df_3l_statistics = df_merged_3l.groupby('image_name').agg(statistic_dict).reset_index()

# Spłaszczanie nazw kolumn (np. ('probability', 'mean') -> 'probability_mean')
df_3l_statistics.columns = ['_'.join([str(c) for c in col]).strip('_') for col in df_3l_statistics.columns.values]
df_3l_statistics.rename(columns={'output_image_first': 'output_image'}, inplace=True)

In [ ]:
# ==========================================
# 3. ZAPIS DO NOWEGO PLIKU CSV
# ==========================================
NOWY_STAT_PATH = '/content/drive/MyDrive/NTWI_project/results/df_statistics_3l.csv'
df_3l_statistics.to_csv(NOWY_STAT_PATH, index=False)
print(f"Nowy plik statystyk 3D został pomyślnie zapisany w: {NOWY_STAT_PATH}")
print(f"Kształt wygenerowanej tabeli: {df_3l_statistics.shape} (Pacjenci x Cechy)\n")

Nowy plik statystyk 3D został pomyślnie zapisany w: /content/drive/MyDrive/NTWI_project/results/df_statistics_3l.csv
Kształt wygenerowanej tabeli: (20, 63) (Pacjenci x Cechy)



# ML model testing (Random Forest)

In [ ]:
# ==========================================
# 4. PRZYGOTOWANIE DANYCH I TEST MODELU
# ==========================================
X_test_3l = df_3l_statistics.drop(columns=['image_name', 'output_image'])
y_test_3l = df_3l_statistics['output_image']

# Wczytanie zapisanego modelu statystycznego z kolorami
MODEL_PATH = '/content/drive/MyDrive/NTWI_project/models/statistics_rf.pkl'
rf_model_loaded = joblib.load(MODEL_PATH)

# KLUCZOWE ZABEZPIECZENIE: Wymuszenie identycznej kolejności kolumn, jak przy treningu
if hasattr(rf_model_loaded, "feature_names_in_"):
    X_test_3l = X_test_3l[rf_model_loaded.feature_names_in_]

# Klasyfikacja i ocena
y_pred_3l = rf_model_loaded.predict(X_test_3l)
bacc_3l = balanced_accuracy_score(y_test_3l, y_pred_3l)

print(f"Ostateczny wynik Balanced Accuracy na danych 3-warstwowych (Statystyki + Kolory): {bacc_3l * 100:.2f}%")
print("Macierz pomyłek dla danych 3D:\n", confusion_matrix(y_test_3l, y_pred_3l))

Ostateczny wynik Balanced Accuracy na danych 3-warstwowych (Statystyki + Kolory): 36.67%
Macierz pomyłek dla danych 3D:
 [[ 3  2]
 [13  2]]


# Multiple Instance Learning (top-K) testing

In [ ]:
# ==========================================
# 5. AGREGACJA TOP-50 DO POZIOMU PACJENTA
# ==========================================

MERGED_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged_3l.csv'
df_merged_3l = pd.read_csv(MERGED_DATAFRAME_PATH)

K = 50
top_k_bags_3l = []
labels_3l = []
slide_names_3l = []

grouped = df_merged_3l.groupby('image_name')

for name, group in grouped:
    group_sorted = group.sort_values(by='probability', ascending=False)
    top_k_tiles = group_sorted.head(K)

    label = top_k_tiles['image_output'].iloc[0]
    labels_3l.append(label)
    slide_names_3l.append(name)

    features_cols = ['probability'] + [col for col in group.columns if str(col).isdigit()]
    flat_features = top_k_tiles[features_cols].values.flatten()

    if len(flat_features) < K * len(features_cols):
        padding = np.zeros(K * len(features_cols) - len(flat_features))
        flat_features = np.concatenate((flat_features, padding))

    top_k_bags_3l.append(flat_features)

X_test_3l = np.array(top_k_bags_3l)
y_test_3l = np.array(labels_3l)

# ==========================================
# 6. POBRANIE WYTRENOWANEGO MODELU I TEST
# ==========================================
# Wczytujemy model, który zapisałeś na dysku pod koniec Etapu 1
rf_model_loaded = joblib.load('/content/drive/MyDrive/NTWI_project/models/best_k5_rf.pkl')

# Sprawdzamy jak zamrożony model radzi sobie na nowych danych 3D
y_pred_3l = rf_model_loaded.predict(X_test_3l)
bacc_3l = balanced_accuracy_score(y_test_3l, y_pred_3l)

print(f"\nOstateczny wynik Balanced Accuracy na danych 3-warstwowych: {bacc_3l * 100:.2f}%")
print("Macierz pomyłek dla danych 3D:\n", confusion_matrix(y_test_3l, y_pred_3l))


Ostateczny wynik Balanced Accuracy na danych 3-warstwowych: 40.00%
Macierz pomyłek dla danych 3D:
 [[2 3]
 [9 6]]
